# 02 · Simulate monthly loan performance

**SIMULATED data, driven by REAL rate movements.**

For each of the 42 months from Apr 2023 to Sep 2026, every open loan is in one state:

`current → 30 days past due → 60 → 90 → 120 → default`

- A **current** loan misses a payment with a small monthly chance (the *hazard*).
- A **late** loan either catches up (*cures*) or falls one bucket further behind.
- The hazard goes **up** when the loan's rate has gone up since signing (payment shock at
  renewal, or prime rising for variable loans) and for **lower credit scores**.

**Output:** `data/loan_months.parquet` — one row per loan per month (≈1.7M rows).
Parquet instead of CSV because it is ~10× smaller and keeps the column types.

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import numpy as np
import pandas as pd
import yaml

from src.logger import get_logger

log = get_logger("02_simulate_performance")

config = yaml.safe_load(open("config/pipeline.yml"))
risk = config["risk"]
drivers = config["hazard_drivers"]
window = config["window"]
# A DIFFERENT seed from notebook 01. With the same seed, this notebook's first random numbers
# would be exactly the ones notebook 01 used to pick each loan's product - so in month 1,
# mortgages (picked by low numbers) missed payments far too often and cards (high numbers) never
# did. Found by checking month-1 delinquency by product.
rng = np.random.default_rng(config["seed"] + 1)

loans = pd.read_csv("data/loans.csv", parse_dates=["origination_month", "renewal_month"])
rates = pd.read_csv("data/rates.csv", parse_dates=["month"])

months = pd.date_range(window["performance_start"], window["performance_end"], freq="MS")
log.info(f"{len(loans):,} loans, {len(months)} months: {months[0]:%Y-%m} to {months[-1]:%Y-%m}")

2026-10-17 17:34:23,128 | 02_simulate_performance | INFO | 50,000 loans, 42 months: 2023-04 to 2026-09


## 1. Base monthly chance of missing a payment, per product
To reach default a loan must miss once, then fail to catch up 4 months in a row
(30 → 60 → 90 → 120 → default). A miss in the last 4 months can't reach default in time,
so only 42 − 4 = 38 months count:

$$\text{cumulative default} \approx 38 \times \text{monthly hazard} \times (1 - \text{cure rate})^4$$

We solve this for the monthly hazard that hits each product's published 42-month default
target. Those targets describe *normal* times — real rate rises (section 3) push some products
above target on purpose. That's the effect we want to measure.

In [2]:
MONTHS_TO_DEFAULT = 4
months_that_can_default = len(months) - MONTHS_TO_DEFAULT

base_hazard = {}
for product, settings in risk.items():
    target, cure = settings["target_cum_default_42m"], settings["cure_rate"]
    base_hazard[product] = target / (months_that_can_default * (1 - cure) ** MONTHS_TO_DEFAULT)

pd.DataFrame({
    "target_42m_default": {p: s["target_cum_default_42m"] for p, s in risk.items()},
    "cure_rate": {p: s["cure_rate"] for p, s in risk.items()},
    "monthly_hazard": base_hazard,
})

,target_42m_default,cure_rate,monthly_hazard
mortgage,0.005,0.50,0.002105
heloc,0.008,0.45,0.002301
auto,0.030,0.40,0.006092
card,0.050,0.30,0.005480


In [3]:
loans["base_hazard"] = loans["product"].map(base_hazard)
loans["cure_rate"] = loans["product"].map({p: s["cure_rate"] for p, s in risk.items()})

# Credit score: 100 points below the reference doubles the risk, 100 points above halves it
points_below = config["portfolio"]["credit_score"]["reference"] - loans["credit_score"]
loans["score_multiplier"] = 2 ** (points_below / drivers["score_points_to_double_risk"])

loans.groupby("product")[["base_hazard", "score_multiplier"]].mean().round(4)

,base_hazard,score_multiplier
product,,
auto,0.0061,0.9811
card,0.0055,0.9801
heloc,0.0023,1.0009
mortgage,0.0021,0.9767


## 2. Balances over time
Mortgages and auto loans pay down every month (same payment formula as notebook 01).
HELOC and card balances are held flat — a simplification.

In [4]:
amortization_months = {"mortgage": risk["mortgage"]["amortization_months"], "auto": risk["auto"]["term_months"]}
loans["amortization_months"] = loans["product"].map(amortization_months)  # empty for HELOC / card

r = loans["rate_at_signing"] / 100 / 12
loans["monthly_payment"] = loans["original_balance"] * r / (1 - (1 + r) ** -loans["amortization_months"])

prime_by_month = rates.set_index("month")["prime_rate"]
loans["prime_at_signing"] = loans["origination_month"].map(prime_by_month)

# True/False arrays used inside the monthly loop
is_amortizing = loans["amortization_months"].notna().to_numpy()
is_auto = (loans["product"] == "auto").to_numpy()
is_variable = (loans["rate_type"] == "variable").to_numpy()   # variable mortgages + HELOCs
is_fixed_renewal = loans["renews_in_window"].to_numpy()

loans[["product", "original_balance", "rate_at_signing", "monthly_payment"]].head()

,product,original_balance,rate_at_signing,monthly_payment
0,card,3609.60,14.76,NaN
1,heloc,205171.32,4.63,NaN
2,card,6661.09,16.74,NaN
3,card,1976.81,18.83,NaN
4,mortgage,881433.51,1.42,3492.138116


## 3. The month-by-month simulation
Each month: work out which loans are open, how much each loan's rate has risen since signing,
its hazard, then draw one random number per loan to decide what happens.

In [5]:
STATUS_NAMES = np.array(["current", "30 dpd", "60 dpd", "90 dpd", "120 dpd", "default"])

status = np.zeros(len(loans), dtype=int)   # 0 = current ... 5 = default
gone = np.zeros(len(loans), dtype=bool)     # defaulted in an earlier month
monthly_rows = []

for month in months:
    months_since_signing = (
        (month.year - loans["origination_month"].dt.year) * 12
        + (month.month - loans["origination_month"].dt.month)
    ).to_numpy()

    # Open this month = already signed, not paid off (auto), not defaulted earlier
    is_open = (months_since_signing >= 0) & ~gone
    is_open &= ~(is_auto & (months_since_signing >= risk["auto"]["term_months"]))

    # How much the loan's rate has gone UP since signing (driven by the real rates)
    rate_increase = np.zeros(len(loans))
    renewed = is_fixed_renewal & (loans["renewal_month"] <= month).to_numpy()
    rate_increase[renewed] = (loans["new_rate"] - loans["old_rate"]).to_numpy()[renewed]
    rate_increase[is_variable] = (prime_by_month[month] - loans["prime_at_signing"]).to_numpy()[is_variable]
    rate_increase = rate_increase.clip(min=0)  # rate drops don't lower risk below the base

    hazard = (
        loans["base_hazard"].to_numpy()
        * loans["score_multiplier"].to_numpy()
        * (1 + drivers["rate_increase_per_pp"] * rate_increase)
    )

    # One random number per loan decides what happens this month
    draw = rng.random(len(loans))
    is_current = is_open & (status == 0)
    is_late = is_open & (status >= 1) & (status <= 4)
    cures = is_late & (draw < loans["cure_rate"].to_numpy())

    new_status = status.copy()
    new_status[is_current & (draw < hazard)] = 1   # misses a payment
    new_status[cures] = 0                          # catches up
    new_status[is_late & ~cures] += 1              # falls further behind
    status = new_status

    # Balance this month
    r = loans["rate_at_signing"].to_numpy() / 100 / 12
    growth = (1 + r) ** months_since_signing
    paid_down = loans["original_balance"].to_numpy() * growth - loans["monthly_payment"].to_numpy() * (growth - 1) / r
    balance = np.where(is_amortizing, paid_down, loans["original_balance"].to_numpy()).clip(min=0)

    monthly_rows.append(pd.DataFrame({
        "loan_id": loans["loan_id"].to_numpy()[is_open],
        "month": month,
        "status": STATUS_NAMES[status[is_open]],
        "balance": balance[is_open].round(2),
        "rate_increase_pp": rate_increase[is_open].round(2),
    }))

    gone |= is_open & (status == 5)  # a defaulted loan leaves the book after its default month

loan_months = pd.concat(monthly_rows, ignore_index=True)
log.info(f"Simulated {len(loan_months):,} loan-month rows")

2026-10-17 17:34:23,690 | 02_simulate_performance | INFO | Simulated 1,688,432 loan-month rows


## 4. Store amounts, not ratios
We save `delinquent_balance` etc. as **dollar amounts**. Rates like "delinquency rate" are
computed later as `SUM(delinquent_balance) / SUM(balance)` so they stay correct when the
dashboard is filtered by province or product. (An average of per-loan ratios would weight a
$5,000 card the same as a $700,000 mortgage.)

In [6]:
late_30_to_120 = loan_months["status"].isin(["30 dpd", "60 dpd", "90 dpd", "120 dpd"])
late_90_plus = loan_months["status"].isin(["90 dpd", "120 dpd"])
loan_months["delinquent_balance"] = loan_months["balance"].where(late_30_to_120, 0)
loan_months["impaired_balance"] = loan_months["balance"].where(late_90_plus, 0)
loan_months["defaulted_balance"] = loan_months["balance"].where(loan_months["status"] == "default", 0)
loan_months["source"] = "SIMULATED - not real customers"

loan_months.to_parquet("data/loan_months.parquet", index=False)
log.info(f"Saved data/loan_months.parquet: {len(loan_months):,} rows")
loan_months.head()

2026-10-17 17:34:24,287 | 02_simulate_performance | INFO | Saved data/loan_months.parquet: 1,688,432 rows


,loan_id,month,status,balance,rate_increase_pp,delinquent_balance,impaired_balance,defaulted_balance,source
0,1,2023-04-01,current,3609.60,0.00,0.0,0.0,0.0,SIMULATED - not real customers
1,2,2023-04-01,current,205171.32,2.75,0.0,0.0,0.0,SIMULATED - not real customers
2,5,2023-04-01,current,648993.34,4.00,0.0,0.0,0.0,SIMULATED - not real customers
3,8,2023-04-01,current,14733.32,0.00,0.0,0.0,0.0,SIMULATED - not real customers
4,9,2023-04-01,current,623781.36,0.00,0.0,0.0,0.0,SIMULATED - not real customers


## 5. Sanity checks
**Check 1 — did products land near their default targets, in the right order?**
Among loans already open in Apr 2023, what share defaulted by Sep 2026?
Expect mortgage < HELOC < auto < card. Expect HELOC *above* target (prime jumped ~4pp in
2023) and auto *below* (many auto loans are paid off partway through the window).

In [7]:
open_at_start = loan_months.loc[loan_months["month"] == months[0], "loan_id"]
defaulted_ids = loan_months.loc[loan_months["status"] == "default", "loan_id"]
check = loans[loans["loan_id"].isin(open_at_start)].copy()
check["defaulted"] = check["loan_id"].isin(defaulted_ids)

result = check.groupby("product")["defaulted"].mean().rename("simulated").to_frame()
result["target"] = pd.Series({p: s["target_cum_default_42m"] for p, s in risk.items()})
(result * 100).round(2).sort_values("simulated")

,simulated,target
product,,
mortgage,0.57,0.5
heloc,1.40,0.8
auto,1.85,3.0
card,4.31,5.0


**Check 2 — follow one loan through time.** Aggregates hide bugs; one traced loan exposes them.
Here: the first fixed mortgage that renewed before Sep 2026. Its `rate_increase_pp` should be 0
before the renewal month and jump to (new rate − old rate) from the renewal month on, and its
balance should keep falling.

In [8]:
renewed_in_data = loans["renews_in_window"] & (loans["renewal_month"] <= months[-1])
one_loan = loans[renewed_in_data].iloc[0]
print(f"Loan {one_loan.loan_id}: renews {one_loan.renewal_month:%Y-%m}, "
      f"old rate {one_loan.old_rate}% -> new rate {one_loan.new_rate}%")
trace = loan_months[loan_months["loan_id"] == one_loan.loan_id]
trace[(trace["month"] >= one_loan.renewal_month - pd.DateOffset(months=2))].head(5)

Loan 28: renews 2025-02, old rate 2.71% -> new rate 4.15%


,loan_id,month,status,balance,rate_increase_pp,delinquent_balance,impaired_balance,defaulted_balance,source
746050,28,2024-12-01,current,682593.85,0.00,0.0,0.0,0.0,SIMULATED - not real customers
786138,28,2025-01-01,current,680471.01,0.00,0.0,0.0,0.0,SIMULATED - not real customers
826510,28,2025-02-01,current,678343.38,1.44,0.0,0.0,0.0,SIMULATED - not real customers
867138,28,2025-03-01,current,676210.94,1.44,0.0,0.0,0.0,SIMULATED - not real customers
908024,28,2025-04-01,current,674073.69,1.44,0.0,0.0,0.0,SIMULATED - not real customers
